# March Machine Learning Mania 2026

This notebook shares the same code path as the scripts under `competitions/march_machine_learning_mania_2026/`.

Before running it, download the competition files:

```bash
python competitions/march_machine_learning_mania_2026/scripts/download_data.py
```


In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'competitions').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from notebook cwd.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

PROJECT_ROOT


In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

from competitions.march_machine_learning_mania_2026.models.baseline import (
    build_datasets,
    default_holdout_season,
    discover_competition_files,
    fit_and_score_holdout,
    fit_final_model,
    generate_submission,
)

sns.set_theme(style='whitegrid')


In [ ]:
raw_dir = PROJECT_ROOT / 'competitions' / 'march_machine_learning_mania_2026' / 'data' / 'raw'
files = discover_competition_files(raw_dir)
training_frame, submission_frame, team_stats = build_datasets(files)

training_frame.head()


In [ ]:
team_stats.groupby(['gender', 'season']).size().rename('teams').reset_index().tail(10)


In [ ]:
plot_df = team_stats[['gender', 'season', 'win_pct', 'elo_rating', 'seed_rank']].copy()
plot_df['season'] = plot_df['season'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(data=plot_df, x='gender', y='win_pct', ax=axes[0])
axes[0].set_title('Regular-season win percentage by gender')
sns.scatterplot(data=plot_df, x='seed_rank', y='elo_rating', hue='gender', alpha=0.6, ax=axes[1])
axes[1].set_title('Seed rank vs Elo rating')
plt.tight_layout()


In [ ]:
holdout_season = default_holdout_season(training_frame, target_season=2026)
_, holdout_metrics, holdout_predictions = fit_and_score_holdout(
    training_frame=training_frame,
    target_season=2026,
    holdout_season=holdout_season,
)

holdout_metrics


In [ ]:
holdout_predictions.head()


In [ ]:
final_model = fit_final_model(training_frame, target_season=2026)
submission = generate_submission(final_model, submission_frame)
submission.head()


In [ ]:
submission_path = PROJECT_ROOT / 'competitions' / 'march_machine_learning_mania_2026' / 'submissions' / 'notebook_submission_preview.csv'
submission.to_csv(submission_path, index=False)
submission_path
